# BMTaddCAT validation

`brand_model_trim_reference.json`에서 실제 `(brand, model_name, trim_name)` 후보를 가져와서
`BMTaddCAT`이 `major_category`를 안정적으로 돌려주는지 확인하는 노트북이다.

- positive case: category CSV에 존재하는 모델
- known missing case: reference JSON에는 있지만 category CSV에는 아직 없는 모델

아래 셀을 위에서부터 순서대로 실행하면 된다.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_data_collection_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and candidate.name == "data_collection":
            return candidate
        nested = candidate / "data_collection"
        if (nested / "pyproject.toml").exists():
            return nested
    raise RuntimeError("Could not locate data_collection directory from current working directory.")


DATA_COLLECTION_DIR = find_data_collection_dir(Path.cwd().resolve())
if str(DATA_COLLECTION_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_COLLECTION_DIR))

from brand_model_trim_category_mapping.BMTaddCAT import CategoryStore, get_major_category_from_tuple


REFERENCE_PATH = DATA_COLLECTION_DIR / "name_brand_model_trim_mapping" / "sources" / "brand_model_trim_reference.json"
CATEGORY_PATH = DATA_COLLECTION_DIR / "brand_model_trim_category_mapping" / "data" / "model_category_mapping_table.csv"

store = CategoryStore.from_path(CATEGORY_PATH)
reference_payload = json.loads(REFERENCE_PATH.read_text(encoding="utf-8"))
reference_map = reference_payload["brand_model_trim_map"]

print(f"DATA_COLLECTION_DIR = {DATA_COLLECTION_DIR}")
print(f"REFERENCE_PATH = {REFERENCE_PATH}")
print(f"CATEGORY_PATH = {CATEGORY_PATH}")

In [ ]:
reference_rows: list[dict[str, object]] = []
for brand, model_map in reference_map.items():
    for model_name, trim_names in model_map.items():
        reference_rows.append(
            {
                "brand": brand,
                "model_name": model_name,
                "trim_count": len(trim_names),
                "sample_trims": ", ".join(trim_names[:3]),
                "major_category": store.get_major_category(brand, model_name),
            }
        )

reference_df = pd.DataFrame(reference_rows)
reference_df["is_mapped"] = reference_df["major_category"].notna()

coverage_summary = (
    reference_df.groupby("brand", as_index=False)
    .agg(
        reference_models=("model_name", "count"),
        mapped_models=("is_mapped", "sum"),
    )
)
coverage_summary["missing_models"] = coverage_summary["reference_models"] - coverage_summary["mapped_models"]

display(coverage_summary)
print(f"total reference models = {len(reference_df)}")
print(f"mapped models = {int(reference_df['is_mapped'].sum())}")
print(f"missing models = {int((~reference_df['is_mapped']).sum())}")

display(reference_df.loc[~reference_df["is_mapped"], ["brand", "model_name", "trim_count", "sample_trims"]].head(20))

In [ ]:
def build_sample_cases(
    reference_map: dict[str, dict[str, list[str]]],
    store: CategoryStore,
    *,
    models_per_brand: int = 4,
    trims_per_model: int = 2,
) -> list[dict[str, object]]:
    cases: list[dict[str, object]] = []
    for brand in sorted(reference_map):
        picked = 0
        for model_name in sorted(reference_map[brand]):
            major_category = store.get_major_category(brand, model_name)
            if major_category is None:
                continue

            trim_names = reference_map[brand][model_name] or [None]
            for trim_name in trim_names[:trims_per_model]:
                cases.append(
                    {
                        "brand": brand,
                        "model_name": model_name,
                        "trim_name": trim_name,
                        "expected_category": major_category,
                    }
                )

            picked += 1
            if picked >= models_per_brand:
                break
    return cases


def build_missing_cases(
    reference_df: pd.DataFrame,
    reference_map: dict[str, dict[str, list[str]]],
    *,
    limit: int = 10,
) -> list[dict[str, object]]:
    missing_cases: list[dict[str, object]] = []
    missing_df = reference_df.loc[~reference_df["is_mapped"], ["brand", "model_name"]].head(limit)
    for row in missing_df.to_dict(orient="records"):
        trim_name = None
        trims = reference_map[row["brand"]][row["model_name"]]
        if trims:
            trim_name = trims[0]
        missing_cases.append(
            {
                "brand": row["brand"],
                "model_name": row["model_name"],
                "trim_name": trim_name,
                "expected_category": None,
            }
        )
    return missing_cases


def validate_cases(cases: list[dict[str, object]]) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for case in cases:
        actual = get_major_category_from_tuple(
            case["brand"],
            case["model_name"],
            case.get("trim_name"),
            store=store,
        )
        rows.append(
            {
                **case,
                "actual_category": actual,
                "passed": actual == case["expected_category"],
            }
        )
    return pd.DataFrame(rows)


sample_cases = build_sample_cases(reference_map, store, models_per_brand=4, trims_per_model=2)
sample_cases_df = pd.DataFrame(sample_cases)
display(sample_cases_df)

In [ ]:
sample_results = validate_cases(sample_cases)
display(sample_results)
assert sample_results["passed"].all(), "Some mapped sample cases failed."
print(f"mapped sample cases passed = {int(sample_results['passed'].sum())} / {len(sample_results)}")

missing_cases = build_missing_cases(reference_df, reference_map, limit=10)
missing_results = validate_cases(missing_cases)
display(missing_results)
assert missing_results["passed"].all(), "Some known-missing cases returned unexpected categories."
print(f"known missing cases passed = {int(missing_results['passed'].sum())} / {len(missing_results)}")

In [ ]:
# 아래 케이스를 직접 수정해서 수동 검증에 사용하면 된다.
manual_cases = [
    {
        "brand": "chevrolet",
        "model_name": "더 넥스트 스파크",
        "trim_name": "LT",
        "expected_category": "hatchback",
    },
    {
        "brand": "kgm",
        "model_name": "G4 렉스턴",
        "trim_name": "마제스티",
        "expected_category": store.get_major_category("kgm", "G4 렉스턴"),
    },
    {
        "brand": "kia",
        "model_name": "EV3",
        "trim_name": "GT 라인",
        "expected_category": store.get_major_category("kia", "EV3"),
    },
]

manual_results = validate_cases(manual_cases)
display(manual_results)